<a href="https://colab.research.google.com/github/marcplanas11-alt/insurance-claims-triage-ai/blob/main/notebooks/claims_agentic_workflow_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

state → intake_agent → policy_agent → decision_agent → validation_agent → escalation_rules

Intake Agent
   ↓
Policy Agent
   ↓
Decision Agent
   ↓
Validation Agent
   ↓
Escalation Router


In [2]:
!pip install anthropic pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 15.6 MB/s eta 0:00:00


In [3]:
import json
from datetime import datetime
import pandas as pd

State o memoria del workflow


In [4]:
def create_initial_state(claim_text, policy_text):
    return {
        "claim_text": claim_text,
        "policy_text": policy_text,
        "intake": None,
        "coverage_analysis": None,
        "decision": None,
        "confidence": None,
        "risk_flags": [],
        "validation": None,
        "human_review_required": False,
        "final_route": None,
        "audit_trail": []
    }

In [5]:
claim_text = """
The insured reports water damage to the kitchen caused by a burst pipe.
Estimated repair cost is £8,500.
The incident occurred 18 days ago.
The insured has provided photos but no plumber invoice yet.
"""

policy_text = """
The policy covers sudden and accidental escape of water from fixed domestic installations.
Claims must be notified as soon as reasonably practicable.
Gradual damage, wear and tear, and pre-existing damage are excluded.
The insurer may request invoices, photos, and repair reports.
"""

state = create_initial_state(claim_text, policy_text)
state

{'claim_text': '\nThe insured reports water damage to the kitchen caused by a burst pipe.\nEstimated repair cost is £8,500.\nThe incident occurred 18 days ago.\nThe insured has provided photos but no plumber invoice yet.\n',
 'policy_text': '\nThe policy covers sudden and accidental escape of water from fixed domestic installations.\nClaims must be notified as soon as reasonably practicable.\nGradual damage, wear and tear, and pre-existing damage are excluded.\nThe insurer may request invoices, photos, and repair reports.\n',
 'intake': None,
 'coverage_analysis': None,
 'decision': None,
 'confidence': None,
 'risk_flags': [],
 'validation': None,
 'human_review_required': False,
 'final_route': None,
 'audit_trail': []}

audit trail


In [6]:
def add_audit_event(state, step, message):
    event = {
        "timestamp": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC"),
        "step": step,
        "message": message
    }
    state["audit_trail"].append(event)
    return state

In [7]:
from datetime import datetime, UTC

def add_audit_event(state, step, message):
    event = {
        "timestamp": datetime.now(UTC).strftime("%Y-%m-%d %H:%M:%S UTC"),
        "step": step,
        "message": message
    }
    state["audit_trail"].append(event)
    return state

In [8]:
state = add_audit_event(
    state,
    step="Test fix",
    message="Using timezone-aware timestamp"
)

state["audit_trail"]

[{'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Test fix',
  'message': 'Using timezone-aware timestamp'}]

Intake Agent

In [9]:
def intake_agent(state):
    claim = state["claim_text"].lower()

    intake_result = {
        "claim_type": None,
        "cause": None,
        "estimated_loss": None,
        "risk_flags": []
    }

    # Tipo de claim
    if "water" in claim or "pipe" in claim:
        intake_result["claim_type"] = "water_damage"

    # Causa
    if "burst pipe" in claim:
        intake_result["cause"] = "burst_pipe"

    # Importe
    if "8500" in claim or "8,500" in claim:
        intake_result["estimated_loss"] = 8500

    # Flags básicos
    if "18 days" in claim:
        intake_result["risk_flags"].append("late_notification")

    # 🔧 NUEVA LÓGICA CORRECTA
    missing_doc_terms = [
        "no invoice",
        "no plumber invoice",
        "missing invoice",
        "no repair report",
        "missing documentation",
        "no supporting documents"
    ]

    if any(term in claim for term in missing_doc_terms):
        intake_result["risk_flags"].append("missing_documentation")

    # Guardar en state
    state["intake"] = intake_result
    state["risk_flags"] = list(set(state["risk_flags"] + intake_result["risk_flags"]))

    # Audit
    state = add_audit_event(
        state,
        step="Intake Agent",
        message=f"Extracted claim type: {intake_result['claim_type']}, flags: {intake_result['risk_flags']}"
    )

    return state

In [10]:
state = intake_agent(state)

state["intake"]

{'claim_type': 'water_damage',
 'cause': 'burst_pipe',
 'estimated_loss': 8500,
 'risk_flags': ['late_notification', 'missing_documentation']}

In [11]:
state["risk_flags"]

['late_notification', 'missing_documentation']

In [12]:
state["audit_trail"]

[{'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Test fix',
  'message': 'Using timezone-aware timestamp'},
 {'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Intake Agent',
  'message': "Extracted claim type: water_damage, flags: ['late_notification', 'missing_documentation']"}]

Policy Agent

In [13]:
def policy_agent(state):
    policy = state["policy_text"].lower()
    intake = state["intake"]

    coverage_result = {
        "coverage_position": "unclear",
        "covered_peril": None,
        "exclusions": [],
        "conditions": [],
        "missing_documents": [],
        "coverage_confidence": 0.5
    }

    # Covered peril
    if intake["claim_type"] == "water_damage" and "escape of water" in policy:
        coverage_result["coverage_position"] = "potentially_covered"
        coverage_result["covered_peril"] = "escape_of_water"
        coverage_result["coverage_confidence"] = 0.75

    # Exclusions
    if "wear and tear" in policy:
        coverage_result["exclusions"].append("wear_and_tear")

    if "gradual damage" in policy:
        coverage_result["exclusions"].append("gradual_damage")

    # Conditions
    if "notified as soon as reasonably practicable" in policy:
        coverage_result["conditions"].append("prompt_notification_required")

    # Missing documents
    if "missing_documentation" in state["risk_flags"]:
        coverage_result["missing_documents"].append("plumber_invoice")

    # Store in state
    state["coverage_analysis"] = coverage_result

    # Add risk flags
    if coverage_result["coverage_position"] == "unclear":
        state["risk_flags"].append("coverage_unclear")

    if coverage_result["missing_documents"]:
        state["risk_flags"].append("missing_documentation")

    state["risk_flags"] = sorted(list(set(state["risk_flags"])))

    # Audit
    state = add_audit_event(
        state,
        step="Policy Agent",
        message=f"Coverage position: {coverage_result['coverage_position']}, confidence: {coverage_result['coverage_confidence']}"
    )

    return state

In [14]:
state = policy_agent(state)

state["coverage_analysis"]

{'coverage_position': 'potentially_covered',
 'covered_peril': 'escape_of_water',
 'exclusions': ['wear_and_tear', 'gradual_damage'],
 'conditions': ['prompt_notification_required'],
 'missing_documents': ['plumber_invoice'],
 'coverage_confidence': 0.75}

In [15]:
state["risk_flags"]

['late_notification', 'missing_documentation']

In [16]:
state["audit_trail"]

[{'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Test fix',
  'message': 'Using timezone-aware timestamp'},
 {'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Intake Agent',
  'message': "Extracted claim type: water_damage, flags: ['late_notification', 'missing_documentation']"},
 {'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Policy Agent',
  'message': 'Coverage position: potentially_covered, confidence: 0.75'}]

Decision Agent

In [17]:
def decision_agent(state):
    coverage = state["coverage_analysis"]
    flags = set(state["risk_flags"])

    decision_result = {
        "decision": "ESCALATE",
        "confidence": 0.5,
        "reason": "",
        "human_review_recommended": True
    }

    if coverage["coverage_position"] == "potentially_covered":
        decision_result["decision"] = "APPROVE"
        decision_result["confidence"] = coverage["coverage_confidence"]
        decision_result["reason"] = "Claim appears to fall within covered escape of water peril."

    if "missing_documentation" in flags:
        decision_result["decision"] = "ESCALATE"
        decision_result["confidence"] -= 0.15
        decision_result["reason"] += " Missing documentation requires human review."

    if "late_notification" in flags:
        decision_result["decision"] = "ESCALATE"
        decision_result["confidence"] -= 0.10
        decision_result["reason"] += " Late notification may affect claim handling."

    if "coverage_unclear" in flags:
        decision_result["decision"] = "ESCALATE"
        decision_result["confidence"] -= 0.20
        decision_result["reason"] += " Coverage position is unclear."

    decision_result["confidence"] = max(0, round(decision_result["confidence"], 2))
    decision_result["human_review_recommended"] = decision_result["decision"] == "ESCALATE"

    state["decision"] = decision_result
    state["confidence"] = decision_result["confidence"]

    state = add_audit_event(
        state,
        step="Decision Agent",
        message=f"Decision: {decision_result['decision']}, confidence: {decision_result['confidence']}"
    )

    return state

In [18]:
state = decision_agent(state)

state["decision"]

{'decision': 'ESCALATE',
 'confidence': 0.5,
 'reason': 'Claim appears to fall within covered escape of water peril. Missing documentation requires human review. Late notification may affect claim handling.',
 'human_review_recommended': True}

In [19]:
state["confidence"]

0.5

In [20]:
state["audit_trail"]

[{'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Test fix',
  'message': 'Using timezone-aware timestamp'},
 {'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Intake Agent',
  'message': "Extracted claim type: water_damage, flags: ['late_notification', 'missing_documentation']"},
 {'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Policy Agent',
  'message': 'Coverage position: potentially_covered, confidence: 0.75'},
 {'timestamp': '2026-04-28 10:27:36 UTC',
  'step': 'Decision Agent',
  'message': 'Decision: ESCALATE, confidence: 0.5'}]

In [21]:
state["decision"]

{'decision': 'ESCALATE',
 'confidence': 0.5,
 'reason': 'Claim appears to fall within covered escape of water peril. Missing documentation requires human review. Late notification may affect claim handling.',
 'human_review_recommended': True}

In [22]:
state["risk_flags"]

['late_notification', 'missing_documentation']

In [25]:
# Definimos las variables necesarias que faltaban
claim = state["claim_text"].lower()
intake_result = {"risk_flags": []}

missing_doc_terms = [
    "no invoice",
    "no plumber invoice",
    "missing invoice",
    "no repair report",
    "missing documentation",
    "no supporting documents"
]

if any(term in claim for term in missing_doc_terms):
    intake_result["risk_flags"].append("missing_documentation")

print(f"Flags detectados: {intake_result['risk_flags']}")

Flags detectados: ['missing_documentation']


Validation Agent

In [26]:
def validation_agent(state):
    decision = state["decision"]
    flags = set(state["risk_flags"])

    validation_result = {
        "validation_status": "PASS",
        "issues": [],
        "confidence_adjustment": 0.0,
        "human_review_required": False,
        "reason": ""
    }

    # Si hay documentación faltante
    if "missing_documentation" in flags:
        validation_result["validation_status"] = "REVIEW_REQUIRED"
        validation_result["issues"].append("Missing documentation")
        validation_result["confidence_adjustment"] -= 0.15
        validation_result["human_review_required"] = True

    # Si hay notificación tardía
    if "late_notification" in flags:
        validation_result["validation_status"] = "REVIEW_REQUIRED"
        validation_result["issues"].append("Late notification")
        validation_result["confidence_adjustment"] -= 0.10

    # Si cobertura no está clara
    if "coverage_unclear" in flags:
        validation_result["validation_status"] = "FAIL"
        validation_result["issues"].append("Coverage unclear")
        validation_result["confidence_adjustment"] -= 0.20
        validation_result["human_review_required"] = True

    # Ajustar confianza
    state["confidence"] = max(0, round(state["confidence"] + validation_result["confidence_adjustment"], 2))

    state["validation"] = validation_result

    # Forzar revisión humana si aplica
    if validation_result["human_review_required"]:
        state["human_review_required"] = True

    # Audit
    state = add_audit_event(
        state,
        step="Validation Agent",
        message=f"Validation: {validation_result['validation_status']}, issues: {validation_result['issues']}"
    )

    return state

In [27]:
state = validation_agent(state)

state["validation"], state["confidence"]

({'validation_status': 'REVIEW_REQUIRED',
  'issues': ['Missing documentation', 'Late notification'],
  'confidence_adjustment': -0.25,
  'human_review_required': True,
  'reason': ''},
 0.25)

In [28]:
state["human_review_required"]
state["audit_trail"]

[{'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Test fix',
  'message': 'Using timezone-aware timestamp'},
 {'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Intake Agent',
  'message': "Extracted claim type: water_damage, flags: ['late_notification', 'missing_documentation']"},
 {'timestamp': '2026-04-28 10:27:35 UTC',
  'step': 'Policy Agent',
  'message': 'Coverage position: potentially_covered, confidence: 0.75'},
 {'timestamp': '2026-04-28 10:27:36 UTC',
  'step': 'Decision Agent',
  'message': 'Decision: ESCALATE, confidence: 0.5'},
 {'timestamp': '2026-04-28 10:29:01 UTC',
  'step': 'Validation Agent',
  'message': "Validation: REVIEW_REQUIRED, issues: ['Missing documentation', 'Late notification']"}]

Routing Rules

In [29]:
def routing_rules(state, confidence_threshold=0.70):
    reasons = []

    if state["confidence"] < confidence_threshold:
        state["human_review_required"] = True
        reasons.append(
            f"Confidence {state['confidence']} below threshold {confidence_threshold}"
        )

    if "missing_documentation" in state["risk_flags"]:
        state["human_review_required"] = True
        reasons.append("Missing documentation")

    if "coverage_unclear" in state["risk_flags"]:
        state["human_review_required"] = True
        reasons.append("Coverage unclear")

    if state["human_review_required"]:
        state["final_route"] = "HUMAN_REVIEW"
    else:
        state["final_route"] = "AUTO_PROCESS"

    state = add_audit_event(
        state,
        step="Routing Rules",
        message=f"Final route: {state['final_route']}. Reasons: {reasons}"
    )

    return state

In [30]:
state = routing_rules(state)

state["final_route"], state["human_review_required"]

('HUMAN_REVIEW', True)

In [31]:
pd.DataFrame(state["audit_trail"])

,timestamp,step,message
0,2026-04-28 10:27:35 UTC,Test fix,Using timezone-aware timestamp
1,2026-04-28 10:27:35 UTC,Intake Agent,"Extracted claim type: water_damage, flags: ['l..."
2,2026-04-28 10:27:35 UTC,Policy Agent,"Coverage position: potentially_covered, confid..."
3,2026-04-28 10:27:36 UTC,Decision Agent,"Decision: ESCALATE, confidence: 0.5"
4,2026-04-28 10:29:01 UTC,Validation Agent,"Validation: REVIEW_REQUIRED, issues: ['Missing..."
5,2026-04-28 10:33:18 UTC,Routing Rules,Final route: HUMAN_REVIEW. Reasons: ['Confiden...


In [32]:
summary = {
    "decision": state["decision"],
    "validation": state["validation"],
    "final_confidence": state["confidence"],
    "final_route": state["final_route"],
    "risk_flags": state["risk_flags"],
    "human_review_required": state["human_review_required"]
}

summary

{'decision': {'decision': 'ESCALATE',
  'confidence': 0.5,
  'reason': 'Claim appears to fall within covered escape of water peril. Missing documentation requires human review. Late notification may affect claim handling.',
  'human_review_recommended': True},
 'validation': {'validation_status': 'REVIEW_REQUIRED',
  'issues': ['Missing documentation', 'Late notification'],
  'confidence_adjustment': -0.25,
  'human_review_required': True,
  'reason': ''},
 'final_confidence': 0.25,
 'final_route': 'HUMAN_REVIEW',
 'risk_flags': ['late_notification', 'missing_documentation'],
 'human_review_required': True}